# How far is the nearest clinic?

Travel-time-to-care analysis for Cameroon. This notebook walks through the
pipeline: build a drivable road graph, snap health facilities to it, compute
nearest-facility travel time for each demand cell, and summarise coverage by
admin-2 unit and national total.

The computation lives in `src/access/`; the equity arithmetic is unit-tested
in `tests/`.

## 0. Setup

Make the `src` layout importable and load the config.

In [ ]:
import sys
from pathlib import Path

import yaml

REPO = Path.cwd().parent
sys.path.insert(0, str(REPO / "src"))

cfg = yaml.safe_load((REPO / "config" / "sources.yaml").read_text())
thresholds = cfg["thresholds_minutes"]
proj_crs = cfg["study_area"]["projected_crs"]
thresholds, proj_crs

## 1. Build the road network

Run `python data/download.py` first to fetch the OSM extract into `data/raw/`.
Then build a drivable graph with `travel_time` on every edge.

In [ ]:
from access.network import build_drive_graph

osm_path = REPO / "data" / "raw" / cfg["sources"]["osm"]["filename"]
# graph = build_drive_graph(osm_path)  # requires the OSM download + osmnx
# graph.number_of_nodes(), graph.number_of_edges()

## 2. Facilities and accessibility

Clean + snap facilities, then run the multi-source shortest path over
`travel_time` to assign each demand cell its nearest-facility time.

In [ ]:
from access.facilities import facility_source_nodes, load_facilities, snap_facilities_to_nodes
from access.access import assign_access_to_demand, build_demand_surface

# facilities = snap_facilities_to_nodes(load_facilities(fac_path, proj_crs), graph)
# sources = facility_source_nodes(facilities)
# demand = assign_access_to_demand(build_demand_surface(admin, cfg), graph, sources, proj_crs)
# demand.head()

## 3. Equity summary

The equity arithmetic is pure pandas/numpy; the synthetic frame below runs
without any geospatial inputs and shows the output schema.

In [ ]:
import pandas as pd
from access.equity import national_summary, summarise_by_admin

demo = pd.DataFrame(
    {
        "admin2": ["North", "North", "South", "South"],
        "travel_time_min": [20.0, 95.0, 10.0, 40.0],
        "population": [300.0, 500.0, 200.0, 100.0],
    }
)
summarise_by_admin(demo, thresholds)

In [ ]:
nat = national_summary(demo, thresholds)
share_beyond = nat[f"share_beyond_{thresholds[1]}min"]
print(f"Share living beyond {thresholds[1]} minutes of care: {share_beyond:.0%}")

## 5. Map it

With real outputs, render a choropleth of population-weighted travel time by
admin-2 unit and save it to `outputs/`:

```python
import matplotlib.pyplot as plt
ax = admin_stats.plot(column="share_beyond_60min", legend=True, cmap="Reds")
ax.set_axis_off()
plt.savefig(REPO / "outputs" / "access_choropleth.png", dpi=150, bbox_inches="tight")
```

In [ ]:
from access.demo import run_demo

metrics = run_demo(seed=0, out_dir=REPO / "outputs")
metrics

### Coverage, inequality, and facility load

Read the demand frame the demo wrote and compute the deeper metrics on it: the
population-weighted **Gini** of travel time (0 = everyone equal, higher = more
unequal), the **coverage bands** (disjoint travel-time buckets that partition the
population), and the **facility load** (population assigned to each facility).

In [ ]:
from access.equity import coverage_bands
from access.metrics import facility_load, gini_coefficient

demand_df = pd.read_csv(REPO / "outputs" / "demand.csv")

gini = gini_coefficient(demand_df["travel_time_min"], demand_df["population"])
bands = coverage_bands(demand_df["travel_time_min"], demand_df["population"], thresholds)
load = facility_load(
    demand_df["node_id"], demand_df["nearest_facility"], demand_df["population"]
)

print(f"Gini of travel time (population-weighted): {gini:.3f}")
print(f"Share within 60 min: {metrics['share_within_60min']:.0%}")
print(f"Population unreachable: {metrics['pop_unreachable']:.0f}")
print("Coverage bands:", {k: round(v, 1) for k, v in bands.items() if k.startswith("pop_band")})
print("Facility load:", {k: round(v, 1) for k, v in load.items()})

## 4. Map it

With real outputs, render a choropleth of population-weighted travel time by
admin-2 unit and save it to `outputs/`:

```python
import matplotlib.pyplot as plt
ax = admin_stats.plot(column="share_beyond_60min", legend=True, cmap="Reds")
ax.set_axis_off()
plt.savefig(REPO / "outputs" / "access_choropleth.png", dpi=150, bbox_inches="tight")
```